# 🎙️ Pipeline Skripsi VoxCPM: Residual & Modulasi Speech Deepfake

Jalankan cell di bawah ini **secara berurutan (Shift + Enter)** dari atas ke bawah.

## 1. Setup Environment & Clone Repository

In [ ]:
!git clone https://github.com/alvinrw/skripsi_fase2.git
%cd skripsi_fase2

!pip install -q numpy pandas scipy scikit-learn librosa soundfile xgboost pyyaml joblib tqdm statsmodels matplotlib seaborn
print("✅ Setup environment selesai!")

## 2. Hubungkan Google Drive (Wajib!)
Ini memastikan model dan hasil perhitungan (file CSV) **tidak hilang** saat Colab ditutup, dan juga digunakan untuk membaca dataset mentah VoxCPM Anda.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/skripsi_results'
for folder in ['results', 'checkpoints', 'manifests']:
    os.makedirs(f'{DRIVE_PATH}/{folder}', exist_ok=True)
    if not os.path.islink(folder):
        if os.path.exists(folder):
            !rm -rf {folder}
        os.symlink(f'{DRIVE_PATH}/{folder}', folder)

print(f"✅ Drive terhubung! Semua hasil akan otomatis tersimpan di: {DRIVE_PATH}")

## 2.5. Unduh Dataset Tambahan (Kaggle)
Dataset Kaggle ini akan digabungkan dengan dataset VoxCPM.

In [ ]:
import os
import json
from google.colab import userdata

# Memuat kredensial dari Colab Secrets
try:
    kaggle_username = userdata.get('KAGGLE_USERNAME')
    kaggle_api_token = userdata.get('KAGGLE_KEY')
    
    kaggle_path = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_path, exist_ok=True)
    with open(os.path.join(kaggle_path, 'kaggle.json'), 'w') as f:
        json.dump({"username": kaggle_username, "key": kaggle_api_token}, f)
    os.chmod(os.path.join(kaggle_path, 'kaggle.json'), 0o600)
    print("✅ Kredensial Kaggle berhasil dimuat dari Colab Secrets.")
except Exception as e:
    print("❌ Gagal memuat kredensial Kaggle. Pastikan Anda telah menambahkan KAGGLE_USERNAME dan KAGGLE_KEY di tab Secrets (ikon kunci) di sebelah kiri Colab.")

# Install Kaggle CLI
!pip install -q kaggle

print("Mengunduh dataset...")

# 1. the-fake-or-real-dataset
!kaggle datasets download -d mohammedabdeldayem/the-fake-or-real-dataset --force --unzip -p /content/kaggle_datasets/the-fake-or-real

# 2. deep-voice-deepfake-voice-recognition
!kaggle datasets download -d birdy654/deep-voice-deepfake-voice-recognition --force --unzip -p /content/kaggle_datasets/deep-voice

# 3. deepfake-audio-dataset-fake-vs-real-speech
!kaggle datasets download -d jayjoshi37/deepfake-audio-dataset-fake-vs-real-speech --force --unzip -p /content/kaggle_datasets/jay15k

print("✅ Unduhan selesai.")

## 3. Persiapan Dataset VoxCPM
Melakukan pengecekan kelayakan audio, pemotongan (*chunking*) multi-durasi, dan men-*zip* hasilnya agar I/O lebih ringan.

*(Pastikan dataset mentah Anda berada di `/content/drive/MyDrive/VoxCPM/Suara_real` dan `/content/drive/MyDrive/VoxCPM/output_generate`)*

In [ ]:
!python src/run_pipeline.py --steps prepare --drive_dir "/content/drive/MyDrive/VoxCPM" --out_dir "/content/VoxCPM_processed"  --kaggle_dirs "/content/kaggle_datasets/the-fake-or-real" "/content/kaggle_datasets/deep-voice" "/content/kaggle_datasets/jay15k"

# Hapus dataset raw Kaggle untuk menghemat penyimpanan (Disk) Colab
!rm -rf /content/kaggle_datasets
print("✅ Cache dataset Kaggle telah dihapus untuk menghemat Disk!")


## 4. Eksekusi Penuh Pipeline Skripsi 🔥
Ekstraksi fitur MFCC/LFCC/Residual/Modulasi, *training* algoritma Machine Learning, serta analisis statistik.

**Perhatikan!** Tahap `prepare` di atas sudah otomatis membuat manifest dan membagi split (70/15/15) ke dalam file `manifests/split_manifest_2s.csv` (dan varian durasi lainnya).

> ⏳ **Catatan:** Ekstraksi fitur dan *training* akan memakan waktu cukup lama karena dieksekusi berturut-turut untuk **durasi (2s)**.

In [ ]:
!python src/run_pipeline.py --steps features --duration 2s
!python src/run_pipeline.py --steps train --duration 2s
!python src/run_pipeline.py --steps stats consistency bootstrap --duration 2s
print('✅ Eksekusi durasi 2s selesai!')

## 5. Visualisasi & Perbandingan Hasil 📊
Menyatukan seluruh file `metrics_*.csv` dari folder hasil dan menampilkan perbandingan model mana yang paling bagus di durasi mana.

In [ ]:
import pandas as pd
import glob
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Mencari semua file metrics dari berbagai durasi
metric_files = sorted(glob.glob('results/metrics_*s.csv'))
dfs = []
for f in metric_files:
    dur = f.split('_')[-1].replace('.csv', '')
    df = pd.read_csv(f)
    df['durasi'] = dur
    dfs.append(df)

if not dfs:
    print("Belum ada data metrics yang tersimpan. Pastikan proses training sudah selesai.")
else:
    df_all = pd.concat(dfs, ignore_index=True)
    # Filter hanya data Test untuk hasil yang riil
    df_test = df_all[df_all['split'] == 'test'].copy()
    
    print("🏆 TOP 5 MODEL TERBAIK (Berdasarkan EER Terendah):")
    display(df_test.sort_values('eer').head(5)[['durasi', 'model', 'eer', 'auc', 'f1_macro']])
    
    # Membuat Plot Grafik Batang
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df_test, x='durasi', y='eer', hue='model')
    plt.title('Perbandingan Equal Error Rate (EER) Tiap Model Berdasarkan Durasi Audio')
    plt.ylabel('EER (Lebih rendah = Lebih baik)')
    plt.xlabel('Durasi Potongan Audio')
    plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()


## 🎉 Selesai!
Semua proses telah tuntas. Folder **`skripsi_results/`** di Google Drive Anda sekarang berisi seluruh *checkpoint* model, hasil statistik uji hipotesis, dan tabel metrik lengkap siap pakai untuk ditaruh di naskah laporan Anda!